# Validazione holdout object-centric basata su grafi

Questo notebook valuta la generalizzazione strutturale del comportamento
object-centric mediante le API native di PM4Py per:

- Object-Centric Directly-Follows Graph (OC-DFG);
- Object Type Graph (OTG);
- Event Type - Object Type Graph (ET-OT).

I modelli normativi vengono scoperti esclusivamente dal training e confrontati
con il test. Training e test sono ottenuti separando intere componenti connesse
del grafo di interazione tra gli oggetti strutturali `orders`, `items` e
`packages`.

Questa analisi costituisce un **conformance checking object-centric basato su
grafi**. Non equivale a un replay sincronizzato sulla Object-Centric Petri Net.

In [1]:
import pandas as pd
from IPython.display import display
from pm4py.algo.discovery.ocel.ocdfg import (
    algorithm as ocdfg_discovery,
)

from ocpm_partial_order.config import MAIN_DATASET_DB
from ocpm_partial_order.conformance import (
    evaluate_object_centric_graph_holdout,
)
from ocpm_partial_order.io.ocel_loader import (
    load_ocel2_sqlite,
)

pd.set_option("display.max_colwidth", None)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




## Caricamento del dataset e valutazione

Lo split viene calcolato sulle componenti connesse formate esclusivamente dagli
oggetti strutturali. L'esclusione di `employees` e `products` evita che oggetti
condivisi di tipo risorsa o riferimento colleghino l'intero log in una singola
componente.

In [2]:
ocel = load_ocel2_sqlite(MAIN_DATASET_DB)

evaluation = evaluate_object_centric_graph_holdout(
    ocel=ocel,
    object_types=("orders", "items", "packages"),
    train_ratio=0.80,
)

split = evaluation.split
print("Valutazione completata.")

Valutazione completata.


## Split object-centric senza leakage

Le componenti sono ordinate rispetto al loro primo timestamp. Il punto di
separazione viene scelto minimizzando la distanza dal rapporto di training
richiesto, senza dividere alcuna componente.

In [3]:
split_summary = pd.DataFrame(
    [
        ("Tipi strutturali", ", ".join(split.object_types)),
        ("Componenti totali", split.component_count),
        (
            "Componenti di training",
            split.training_component_count,
        ),
        (
            "Componenti di test",
            split.test_component_count,
        ),
        (
            "Eventi di training",
            split.training_event_count,
        ),
        ("Eventi di test", split.test_event_count),
        (
            "Oggetti di training",
            split.training_object_count,
        ),
        ("Oggetti di test", split.test_object_count),
        (
            "Rapporto effettivo di training",
            round(split.effective_training_ratio, 4),
        ),
        (
            "Eventi condivisi",
            split.shared_event_count,
        ),
        (
            "Oggetti condivisi",
            split.shared_object_count,
        ),
    ],
    columns=["Misura", "Valore"],
)

display(split_summary)

,Misura,Valore
0,Tipi strutturali,"items, orders, packages"
1,Componenti totali,68
2,Componenti di training,44
3,Componenti di test,24
4,Eventi di training,16547
5,Eventi di test,4461
6,Oggetti di training,8531
7,Oggetti di test,2256
8,Rapporto effettivo di training,0.7877
9,Eventi condivisi,0


## Fitness native e fitness strutturali

Le fitness predefinite di PM4Py considerano anche le frequenze. Poiche il
training contiene molti piu eventi del test, tali valori risultano
inevitabilmente ridotti dalle differenze di volume.

Le fitness strutturali usano soglie infinite per le differenze di frequenza e
misurano quindi soltanto la presenza o assenza di attivita, flussi, tipi di
oggetto e relazioni.

In [4]:
fitness_table = pd.DataFrame(
    [
        (
            "OC-DFG",
            evaluation.ocdfg_default_fitness,
            evaluation.ocdfg_structural_fitness,
        ),
        (
            "OTG",
            evaluation.otg_default_fitness,
            evaluation.otg_structural_fitness,
        ),
        (
            "ET-OT",
            evaluation.etot_default_fitness,
            evaluation.etot_structural_fitness,
        ),
    ],
    columns=[
        "Rappresentazione",
        "Fitness predefinita",
        "Fitness strutturale",
    ],
)

display(fitness_table.round(6))

,Rappresentazione,Fitness predefinita,Fitness strutturale
0,OC-DFG,0.492537,0.985075
1,OTG,0.571429,1.000000
2,ET-OT,0.634615,1.000000


## Confronto strutturale simmetrico

La fitness nativa e direzionale: confronta il test reale con il modello
normativo scoperto dal training. Per evitare che eventuali elementi aggiuntivi
vengano trascurati, confrontiamo anche gli insiemi in modo simmetrico.

La copertura del test indica la quota degli elementi del test gia osservata nel
training. L'indice di Jaccard misura la somiglianza complessiva tra i due
insiemi.

In [5]:
def comparison_row(name, comparison):
    return {
        "Struttura": name,
        "Training": len(comparison.training_elements),
        "Test": len(comparison.test_elements),
        "Condivisi": len(comparison.shared_elements),
        "Copertura test": comparison.test_coverage,
        "Copertura training": comparison.training_coverage,
        "Jaccard": comparison.jaccard,
        "Nuovi nel test": len(
            comparison.additional_test_elements
        ),
        "Assenti dal test": len(
            comparison.training_elements_absent_from_test
        ),
    }


structural_table = pd.DataFrame(
    [
        comparison_row(
            "OC-DFG: attivita",
            evaluation.ocdfg_activities,
        ),
        comparison_row(
            "OC-DFG: flussi tipizzati",
            evaluation.ocdfg_typed_flows,
        ),
        comparison_row(
            "OTG: tipi di oggetto",
            evaluation.otg_object_types,
        ),
        comparison_row(
            "OTG: archi",
            evaluation.otg_edges,
        ),
        comparison_row(
            "ET-OT: attivita",
            evaluation.etot_activities,
        ),
        comparison_row(
            "ET-OT: tipi di oggetto",
            evaluation.etot_object_types,
        ),
        comparison_row(
            "ET-OT: relazioni",
            evaluation.etot_relations,
        ),
    ]
)

display(structural_table.round(6))

,Struttura,Training,Test,Condivisi,Copertura test,Copertura training,Jaccard,Nuovi nel test,Assenti dal test
0,OC-DFG: attivita,11,11,11,1.0,1.000000,1.000000,0,0
1,OC-DFG: flussi tipizzati,66,64,64,1.0,0.969697,0.969697,0,2
2,OTG: tipi di oggetto,3,3,3,1.0,1.000000,1.000000,0,0
3,OTG: archi,9,9,9,1.0,1.000000,1.000000,0,0
4,ET-OT: attivita,11,11,11,1.0,1.000000,1.000000,0,0
5,ET-OT: tipi di oggetto,3,3,3,1.0,1.000000,1.000000,0,0
6,ET-OT: relazioni,19,19,19,1.0,1.000000,1.000000,0,0


## Flussi OC-DFG non riprodotti nel test

I flussi vengono confrontati mantenendo il tipo di oggetto. Questo dettaglio e
importante perche la fitness OC-DFG nativa di PM4Py unisce i flussi dei diversi
tipi di oggetto.

In [6]:
training_only_flows = sorted(
    evaluation.ocdfg_typed_flows
    .training_elements_absent_from_test
)

training_ocdfg = ocdfg_discovery.apply(
    split.training_ocel
)

flow_rows = []

for object_type, source, target in training_only_flows:
    event_couples = (
        training_ocdfg["edges"]["event_couples"]
        [object_type][(source, target)]
    )
    unique_objects = (
        training_ocdfg["edges"]["unique_objects"]
        [object_type][(source, target)]
    )
    total_objects = (
        training_ocdfg["edges"]["total_objects"]
        [object_type][(source, target)]
    )

    flow_rows.append(
        {
            "Tipo di oggetto": object_type,
            "Sorgente": source,
            "Destinazione": target,
            "Coppie di eventi": len(event_couples),
            "Oggetti unici": len(unique_objects),
            "Occorrenze sugli oggetti": len(total_objects),
        }
    )

missing_flow_table = pd.DataFrame(flow_rows)

if missing_flow_table.empty:
    print(
        "Tutti i flussi del training sono "
        "riprodotti nel test."
    )
else:
    display(missing_flow_table)

,Tipo di oggetto,Sorgente,Destinazione,Coppie di eventi,Oggetti unici,Occorrenze sugli oggetti
0,items,create package,payment reminder,6,6,6
1,items,payment reminder,send package,6,6,6


## Interpretazione

I risultati mostrano che:

- training e test non condividono eventi oppure oggetti;
- tutte le attivita del test sono presenti nel training;
- tutti i 64 flussi OC-DFG tipizzati del test sono presenti nel training;
- OTG ed ET-OT coincidono strutturalmente tra training e test;
- il test non introduce alcun elemento strutturale nuovo;
- due flussi del training non compaiono nel test e appartengono alla variante
  rara `create package -> payment reminder -> send package`, osservata su sei
  item del training;
- le fitness predefinite piu basse riflettono soprattutto le differenti
  dimensioni dei due insiemi e non devono essere interpretate come una scarsa
  conformita strutturale.

La validazione fornisce quindi evidenza di generalizzazione object-centric
out-of-sample a livello di grafi. Rimane distinto e non ancora implementato un
eventuale replay sincronizzato rispetto alla semantica della OCPN.